<a href="https://colab.research.google.com/github/sebarom06/econ3916-statsml/blob/main/Lab12/Lab_12_OLS%2C_Hedonic_Pricing%2C_RMSE_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.tools.eval_measures import rmse
import plotly.graph_objects as go

# ── Step 1: Synthetic Data ────────────────────────────────────────────────────
np.random.seed(42)
n = 800

square_footage      = np.random.normal(1800, 400, n).clip(600, 4500)
property_age        = np.random.uniform(1, 60, n)
distance_to_transit = np.random.exponential(1.5, n).clip(0.1, 10)

home_value = (
    80_000
    + 120    * square_footage
    - 1_200  * property_age
    - 15_000 * distance_to_transit
    + np.random.normal(0, 18_000, n)
)

df = pd.DataFrame({
    'Home_Value':          home_value,
    'Square_Footage':      square_footage,
    'Property_Age':        property_age,
    'Distance_to_Transit': distance_to_transit,
})

# ── Step 2: OLS Model ─────────────────────────────────────────────────────────
formula = 'Home_Value ~ Square_Footage + Property_Age + Distance_to_Transit'
results = smf.ols(formula=formula, data=df).fit()

model_rmse = rmse(df['Home_Value'], results.predict(df))
print(f"RMSE: ${model_rmse:,.2f}")

# ── Step 3: Residual Extraction ───────────────────────────────────────────────
# results.fittedvalues → the ŷ vector (one value per observation),
# stored as a pandas Series indexed to match df.
y_pred = results.fittedvalues

# results.resid → eᵢ = yᵢ − ŷᵢ for every observation.
# Statsmodels computes this internally during .fit(); we simply retrieve it.
residuals = results.resid

# ── Step 4: Outlier Classification ───────────────────────────────────────────
# Standardise residuals: subtract mean (≈0 by OLS property) and divide by σ.
# Any observation whose |z| > 2 falls outside ~95% of the error distribution
# and is flagged as a structural outlier worth inspecting.
resid_std  = residuals.std()
resid_z    = (residuals - residuals.mean()) / resid_std
is_outlier = resid_z.abs() > 2

# Build a colour vector aligned to the observation index:
# outliers → crimson, normal → steel blue
colors = np.where(is_outlier, '#DC143C', '#4A90D9')

# Hover text — shows key diagnostics per point
hover_text = [
    f"ŷ: ${yp:,.0f}<br>Residual: ${r:,.0f}<br>z: {z:.2f}<br>{'⚠ Outlier' if o else ''}"
    for yp, r, z, o in zip(y_pred, residuals, resid_z, is_outlier)
]

# ── Step 5: Plotly Residual Forensics Plot ────────────────────────────────────
fig = go.Figure()

# Main scatter — all observations, coloured by outlier status
fig.add_trace(go.Scatter(
    x=y_pred,
    y=residuals,
    mode='markers',
    marker=dict(
        color=colors,
        size=5,
        opacity=0.75,
        line=dict(width=0),
    ),
    text=hover_text,
    hoverinfo='text',
    name='Residuals',
))

# Zero-line — the theoretical expectation under homoscedasticity (E[eᵢ] = 0)
fig.add_hline(
    y=0,
    line=dict(color='#F0E68C', width=1.5, dash='dash'),
)

# ±2σ boundary lines — visual envelope for the 95% region
for sign, label in [(1, '+2σ'), (-1, '−2σ')]:
    fig.add_hline(
        y=sign * 2 * resid_std,
        line=dict(color='#888', width=1, dash='dot'),
        annotation_text=label,
        annotation_position='right',
        annotation_font=dict(color='#aaa', size=11),
    )

n_outliers = is_outlier.sum()

fig.update_layout(
    title=dict(
        text=f'Residual Forensics Dashboard'
             f'<br><sup>OLS Hedonic Pricing Model — RMSE ${model_rmse:,.0f} · '
             f'{n_outliers} outliers ({100*n_outliers/n:.1f}%)</sup>',
        font=dict(family='monospace', size=18, color='#e8e4d9'),
    ),
    xaxis=dict(
        title='Fitted Values (ŷ)',
        tickprefix='$',
        tickformat=',.0f',
        gridcolor='#2e2e42',
        color='#aaa',
    ),
    yaxis=dict(
        title='Residuals (y − ŷ)',
        tickprefix='$',
        tickformat=',.0f',
        gridcolor='#2e2e42',
        color='#aaa',
        zeroline=False,
    ),
    plot_bgcolor='#0f0f13',
    paper_bgcolor='#0f0f13',
    font=dict(family='DM Sans, sans-serif', color='#e8e4d9'),
    showlegend=False,
    margin=dict(l=70, r=80, t=90, b=60),
    hoverlabel=dict(
        bgcolor='#1a1a24',
        bordercolor='#2e2e42',
        font=dict(color='#e8e4d9'),
    ),
)

fig.write_html('/mnt/user-data/outputs/residual_dashboard.html')

HTTPError: HTTP Error 400: Bad Request